# TP 12 — RNN (réseau récurrent simple)
**Deep Learning · Séquences**

Ce TP reprend les exemples de la présentation (diapos 82 à 86, et 85 bis) :

| Exemple | Diapo | Problème |
|---|---|---|
| Exemple 1 | 88 | Régression : à partir des ventes de lundi, mardi et mercredi, prédire les ventes de jeudi |
| Exemple 2 | 89 | Classification : les ventes sont-elles en hausse sur ces 3 jours ? |
| Exemple 3 | 90 | Texte : un avis de client est-il positif ? RNN à 2 neurones (Partie D) |

**Le contexte.** Une petite pâtisserie de quartier note chaque soir ses ventes de la journée :

| Jour | Lundi | Mardi | Mercredi | Jeudi |
|---|---|---|---|---|
| Ventes (TND) | 100 | 200 | 300 | **? (réel : 400)** |

Le gérant veut un modèle qui lit les ventes **dans l’ordre des jours** et répond à deux questions :
1. **Combien** vendrai-je jeudi ? → un nombre : **régression** ;
2. Mes ventes sont-elles **en hausse** ? → oui / non : **classification**.

C’est une **série temporelle** : l’ordre des jours compte (100, 200, 300 n’a pas le même sens que 300, 200, 100). C’est exactement ce que sait lire un RNN. On travaille en **centaines de TND** (100 TND → 1) pour garder des calculs simples à la main : la séquence lue par le réseau est donc $1, 2, 3$ et la valeur à prédire $4$.

**Objectifs** : programmer **en Python pur** la lecture de la séquence par la cellule RNN, puis l’entraînement de la couche de sortie (3 epochs) ; refaire le même entraînement avec **Keras** (`SimpleRNN`) ; puis **comparer** les deux implémentations pour valider le code.

## Rappel de cours

Un RNN lit une séquence **pas à pas** et garde une **mémoire** $h_t$ qui résume ce qu’il a déjà lu :
$$h_t = \tanh(W_x\,x_t + W_h\,h_{t-1} + b)$$
Les **mêmes poids** $W_x, W_h, b$ servent à chaque pas de temps (poids partagés). La dernière mémoire $h_3$ alimente une **couche de sortie** :
- régression : $\hat y = v\cdot h + c$, perte $(y - \hat y)^2$ ;
- classification : $p = \sigma(v\cdot h + c)$, perte $-\ln p$ (quand $y = 1$).

**Simplification des diapos** (pour calculer à la main) : $W_x = 1$, $W_h = 0{,}5$, $b = 0$, **sans tanh**, donc
$$h_t = 0{,}5\,h_{t-1} + x_t$$
La cellule est **fixée** : on n’entraîne que la couche de sortie $(v, c)$.

In [ ]:
import math

jours = ["Lundi", "Mardi", "Mercredi"]
ventes_TND = [100, 200, 300]                  # ventes réelles de la pâtisserie
ventes_jeudi_TND = 400                        # ce qu'on aurait voulu prédire

sequence = [v / 100 for v in ventes_TND]      # en centaines de TND : [1.0, 2.0, 3.0]
y_reg = ventes_jeudi_TND / 100                # cible de la régression : 4.0
y_cls = 1                                     # cible de la classification : 1 = ventes en hausse

for j, v, x in zip(jours, ventes_TND, sequence):
    print(f"{j:<9}: {v} TND  →  x = {x}")

Lundi    : 100 TND  →  x = 1.0
Mardi    : 200 TND  →  x = 2.0
Mercredi : 300 TND  →  x = 3.0


---
# Partie A — Implémentation from scratch (Python pur)

### Question 1 — Lire les ventes de lundi, mardi, mercredi avec la cellule RNN
**Réponse.** Le RNN lit un jour à la fois. Il part d’une mémoire vide $h_0 = 0$ (avant lundi, il ne sait rien), puis applique la même formule chaque jour :
- lundi : $h_1 = 0{,}5\times 0 + 1 = 1$
- mardi : $h_2 = 0{,}5\times 1 + 2 = 2{,}5$
- mercredi : $h_3 = 0{,}5\times 2{,}5 + 3 = 4{,}25$

$h_3 = 4{,}25$ résume les trois jours : la vente de mercredi compte entièrement, celle de mardi à moitié, celle de lundi au quart ($3 + 0{,}5\times2 + 0{,}25\times1 = 4{,}25$). Les jours récents pèsent plus que les anciens.

In [ ]:
def lire_sequence(xs, Wx=1.0, Wh=0.5, b=0.0):
    # Cellule RNN simplifiée (sans tanh) : h(t) = Wx·x(t) + Wh·h(t−1) + b
    h = 0.0                       # mémoire vide au départ
    etats = []                    # mémoire après chaque pas
    for x in xs:                  # lecture pas à pas, dans l'ordre
        h = Wx * x + Wh * h + b   # nouvelle mémoire = entrée + une partie de l'ancienne
        etats.append(h)
    return etats

etats = lire_sequence(sequence)
for t, h in enumerate(etats, start=1):
    print(f"t = {t} ({jours[t - 1]:<8}) : x = {sequence[t - 1]} → h{t} = {h:.4f}")

h_final = etats[-1]               # la dernière mémoire résume toute la séquence
print("Mémoire finale transmise à la couche de sortie : h =", h_final)

t = 1 (Lundi   ) : x = 1.0 → h1 = 1.0000
t = 2 (Mardi   ) : x = 2.0 → h2 = 2.5000
t = 3 (Mercredi) : x = 3.0 → h3 = 4.2500
Mémoire finale transmise à la couche de sortie : h = 4.25


### Question 2 — Exemple 1 : prédire les ventes de jeudi (régression, diapo 84)
La couche de sortie est $\hat y = v\cdot h + c$ (en centaines de TND), la perte $(y - \hat y)^2$ avec $y = 4$ (400 TND). Départ $v = c = 0$ ; $\eta = 0{,}02$ ; 3 epochs.

**Réponse.** Erreur en sortie : $\delta = -2(y - \hat y)$. Gradients : $\nabla v = \delta\,h$ et $\nabla c = \delta$. La mémoire $h = 4{,}25$ ne change pas (la cellule est fixée) : seuls $v$ et $c$ bougent.

Epoch 1 à la main : $\hat y = 0$, $\delta = -2\times4 = -8$, donc $v = 0 - 0{,}02\times(-8)\times4{,}25 = 0{,}68$ et $c = 0 - 0{,}02\times(-8) = 0{,}16$.

In [ ]:
def entrainer_regression(h, y, eta, epochs=3):
    v, c = 0.0, 0.0
    lignes = []
    for e in range(epochs + 1):               # epoch 0 = état de départ
        y_chap = v * h + c                    # prédiction
        perte = (y - y_chap) ** 2             # MSE
        lignes.append((e, v, c, y_chap, perte))
        delta = -2 * (y - y_chap)             # dérivée de la perte par rapport à ŷ
        v = v - eta * delta * h               # mise à jour (après la mesure)
        c = c - eta * delta
    return lignes

reg = entrainer_regression(h_final, y=y_reg, eta=0.02)
print("Epoch |    v       c    |   ŷ      perte")
for e, v, c, y_chap, perte in reg:
    print(f"  {e}   | {v:6.3f}  {c:6.3f} | {y_chap:6.3f}  {perte:7.3f}")
print(f"Prévision pour jeudi : {reg[-1][3] * 100:.0f} TND (réel : {ventes_jeudi_TND} TND)")

Epoch |    v       c    |   ŷ      perte
  0   |  0.000   0.000 |  0.000   16.000
  1   |  0.680   0.160 |  3.050    0.902
  2   |  0.842   0.198 |  3.774    0.051
  3   |  0.880   0.207 |  3.946    0.003
Prévision pour jeudi : 395 TND (réel : 400 TND)


**Lecture.** La prévision pour jeudi passe de 0 à 3,95, soit **395 TND** (réel : 400 TND), et la perte de 16 à presque 0.

### Question 3 — Exemple 2 : les ventes sont-elles en hausse ? (classification, diapo 85)
De 100 à 200 puis 300 TND, les ventes montent : la réponse attendue est « oui » ($y = 1$). Sortie $p = \sigma(v\cdot h + c)$, perte $-\ln p$ ; $\delta = p - y$, $\nabla v = \delta\,h$, $\nabla c = \delta$ ; $\eta = 0{,}1$.

**Réponse.** Même mémoire $h$, mais une **tête de classification**. Epoch 1 à la main : $p = \sigma(0) = 0{,}5$, $\delta = -0{,}5$, donc $v = 0{,}1\times0{,}5\times4{,}25 = 0{,}21$ et $c = 0{,}05$.

In [ ]:
def sigmoide(z):
    return 1 / (1 + math.exp(-z))

def entrainer_classification(h, y, eta, epochs=3):
    v, c = 0.0, 0.0
    lignes = []
    for e in range(epochs + 1):
        p = sigmoide(v * h + c)                               # probabilité « ventes en hausse »
        perte = -math.log(p) if y == 1 else -math.log(1 - p)  # log-loss
        lignes.append((e, v, c, p, perte))
        delta = p - y                                         # erreur en sortie
        v = v - eta * delta * h
        c = c - eta * delta
    return lignes

cls = entrainer_classification(h_final, y=y_cls, eta=0.1)
print("Epoch |    v       c    |   p      perte")
for e, v, c, p, perte in cls:
    print(f"  {e}   | {v:6.3f}  {c:6.3f} | {p:6.3f}  {perte:7.3f}")

Epoch |    v       c    |   p      perte
  0   |  0.000   0.000 |  0.500    0.693
  1   |  0.213   0.050 |  0.722    0.326
  2   |  0.331   0.078 |  0.815    0.204
  3   |  0.409   0.096 |  0.862    0.148


**Lecture.** La probabilité que les ventes soient en hausse monte de 0,5 à 0,86 et la perte baisse de 0,69 à 0,15. La même mémoire $h$ sert aux deux tâches : seule la couche de sortie change.

---
# Partie B — Le même entraînement avec Keras

### Question 4 — Construire la cellule RNN dans Keras
**Réponse.** Keras possède la couche `SimpleRNN`. Avec `activation=None` (pas de tanh), elle calcule exactement $h_t = W_x x_t + W_h h_{t-1} + b$. On impose $W_x = 1$, $W_h = 0{,}5$, $b = 0$ et on la **gèle** (`trainable=False`) : seule la couche `Dense` de sortie est entraînée, comme dans la diapo.

Forme des données pour un RNN : `(nombre de séquences, nombre de pas, valeurs par pas)`. Ici une seule séquence (la semaine de la pâtisserie), 3 jours, une vente par jour, donc `(1, 3, 1)`.

In [ ]:
import numpy as np
import keras
from keras import layers

X = np.array(sequence).reshape(1, 3, 1)   # [[[1.], [2.], [3.]]] : 1 séquence, 3 jours, 1 vente par jour

def creer_cellule():
    cellule = layers.SimpleRNN(1, activation=None, trainable=False)   # 1 neurone de mémoire, sans tanh, gelé
    cellule.build((None, 3, 1))
    cellule.set_weights([np.array([[1.0]]),     # W_x (entrée)
                         np.array([[0.5]]),     # W_h (mémoire)
                         np.array([0.0])])      # b
    return cellule

cellule = creer_cellule()
h_keras = float(keras.ops.convert_to_numpy(cellule(X))[0, 0])
print("Mémoire h calculée par Keras :", round(h_keras, 4))

Mémoire h calculée par Keras : 4.25


### Question 5 — Entraîner les deux têtes avec Keras
**Réponse.** Le modèle complet est `Input(3, 1)` → `SimpleRNN` (gelé) → `Dense(1)`. La couche `Dense` part de $v = c = 0$ (`zeros`). On utilise `SGD` avec le même η et la même perte qu’en Python pur : `mse` pour la régression, `binary_crossentropy` pour la classification. Un *callback* enregistre $(v, c)$ au départ puis à la fin de chaque epoch.

In [ ]:
class HistoriqueSortie(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.vc = [self.lire_vc()]                 # (v, c) au départ
    def on_epoch_end(self, epoch, logs=None):
        self.vc.append(self.lire_vc())             # (v, c) après chaque epoch
    def lire_vc(self):
        k, c = self.model.layers[-1].get_weights() # dernière couche = Dense de sortie
        return float(k[0, 0]), float(c[0])

def entrainer_keras(y, eta, classification):
    model = keras.Sequential([
        keras.Input(shape=(3, 1)),
        creer_cellule(),
        layers.Dense(1, activation="sigmoid" if classification else None,
                     kernel_initializer="zeros", bias_initializer="zeros")])
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=eta),
                  loss="binary_crossentropy" if classification else "mse")
    hist = HistoriqueSortie()
    model.fit(X, np.array([[y]]), epochs=3, batch_size=1, verbose=0, callbacks=[hist])
    return model, hist.vc

m_reg, vc_reg = entrainer_keras(y_reg, eta=0.02, classification=False)
m_cls, vc_cls = entrainer_keras(float(y_cls), eta=0.1, classification=True)

print("Régression     : (v, c) par epoch =", [tuple(round(a, 3) for a in t) for t in vc_reg])
print(f"                 prévision pour jeudi = {float(m_reg.predict(X, verbose=0)[0, 0]) * 100:.0f} TND")
print("Classification : (v, c) par epoch =", [tuple(round(a, 3) for a in t) for t in vc_cls])
print("                 p(ventes en hausse) =", round(float(m_cls.predict(X, verbose=0)[0, 0]), 3))

Régression     : (v, c) par epoch = [(0.0, 0.0), (0.68, 0.16), (0.842, 0.198), (0.88, 0.207)]
                 prévision pour jeudi = 395 TND
Classification : (v, c) par epoch = [(0.0, 0.0), (0.213, 0.05), (0.331, 0.078), (0.409, 0.096)]
                 p(ventes en hausse) = 0.862


---
# Partie C — Comparaison et validation

### Question 6 — Python pur et Keras donnent-ils les mêmes résultats ?
**Réponse.** On compare la mémoire $h$, puis $v$ et $c$ à chaque epoch, pour les deux exemples.

In [ ]:
import pandas as pd

lignes = []
for nom, pur, ker in [("régression", reg, vc_reg), ("classification", cls, vc_cls)]:
    for (e, v, c, _, _), (vk, ck) in zip(pur, ker):
        lignes.append((nom, e, v, vk, c, ck))
tableau = pd.DataFrame(lignes, columns=["Exemple", "Epoch", "v Python pur", "v Keras", "c Python pur", "c Keras"])
print(f"Mémoire h : Python pur = {h_final:.4f} ; Keras = {h_keras:.4f}")
display(tableau.round(4))

assert abs(h_keras - h_final) < 1e-4
assert np.allclose([(v, c) for _, v, c, _, _ in reg], vc_reg, atol=1e-4)
assert np.allclose([(v, c) for _, v, c, _, _ in cls], vc_cls, atol=1e-4)
print("✅ Python pur = Keras : même mémoire h et mêmes paramètres de sortie à chaque epoch")

Mémoire h : Python pur = 4.2500 ; Keras = 4.2500


,Exemple,Epoch,v Python pur,v Keras,c Python pur,c Keras
0,régression,0,0.0000,0.0000,0.0000,0.0000
1,régression,1,0.6800,0.6800,0.1600,0.1600
2,régression,2,0.8415,0.8415,0.1980,0.1980
3,régression,3,0.8799,0.8799,0.2070,0.2070
4,classification,0,0.0000,0.0000,0.0000,0.0000
5,classification,1,0.2125,0.2125,0.0500,0.0500
6,classification,2,0.3308,0.3308,0.0778,0.0778
7,classification,3,0.4093,0.4093,0.0963,0.0963


✅ Python pur = Keras : même mémoire h et mêmes paramètres de sortie à chaque epoch


---
# Partie D — Exemple 3 : un RNN à 2 neurones qui lit du texte

### Le problème
La pâtisserie reçoit des **avis de clients**. On veut savoir si un avis est **positif** ($y = 1$) ou **négatif** ($y = 0$) :

| Avis | $y$ |
|---|---|
| gâteau délicieux | 1 |
| service lent | 0 |
| prix cher mais gâteau délicieux | 1 |
| gâteau délicieux mais service lent | 0 |
| accueil sympa et pain frais | 1 |
| pain sec et prix cher | 0 |

Les avis 3 et 4 contiennent un mot positif **et** un mot négatif : c’est **ce qui vient après « mais »** qui décide. Il faut donc lire les mots **dans l’ordre** : c’est un travail pour un RNN.

### Du texte aux nombres : l’embedding
Un RNN ne lit que des nombres. Chaque mot devient un petit vecteur à 2 cases, $x = [\text{positif} ; \text{négatif}]$ :
- mots positifs (délicieux, sympa, frais, bon) : $x = [1 ; 0]$
- mots négatifs (lent, cher, sec, nul) : $x = [0 ; 1]$
- autres mots (gâteau, service, mais, et…) : $x = [0 ; 0]$

Dans un vrai modèle, ces vecteurs (l’**embedding**) sont appris et ont 50 à 300 cases ; ici on les fixe pour pouvoir calculer à la main.

### Le RNN à 2 neurones
Chaque neurone a sa propre mémoire, avec la même règle que dans l’exemple des ventes :
$$h_1 \leftarrow 0{,}5\,h_1 + x_{\text{positif}} \qquad h_2 \leftarrow 0{,}5\,h_2 + x_{\text{négatif}}$$
- le **neurone 1** retient les mots **positifs** ;
- le **neurone 2** retient les mots **négatifs** ;
- à chaque nouveau mot, l’ancienne mémoire est divisée par 2 : **les derniers mots comptent plus**.

En matrices : $W_x = \begin{pmatrix}1&0\\0&1\end{pmatrix}$, $W_h = \begin{pmatrix}0{,}5&0\\0&0{,}5\end{pmatrix}$, $b = 0$. Dans un vrai RNN, ces poids sont appris et chaque neurone **trouve seul** son rôle ; ici on les fixe pour voir clairement ce que fait chaque neurone. La sortie $p = \sigma(v_1 h_1 + v_2 h_2 + c)$ est entraînée.

### Question 7 — Lire un avis mot par mot (Python pur)
**Réponse.** On suit les deux mémoires pour l’avis « gâteau délicieux mais service lent ».

In [ ]:
POSITIFS = {"délicieux", "sympa", "frais", "bon"}
NEGATIFS = {"lent", "cher", "sec", "nul"}

def embedding(mot):
    # mot → vecteur [positif ; négatif]
    return [1.0 if mot in POSITIFS else 0.0, 1.0 if mot in NEGATIFS else 0.0]

def lire_avis(avis, detail=False):
    h1, h2 = 0.0, 0.0                       # mémoires vides au départ
    for mot in avis.split():                # lecture mot par mot, dans l'ordre
        x = embedding(mot)
        h1 = 0.5 * h1 + x[0]                # neurone 1 : mots positifs
        h2 = 0.5 * h2 + x[1]                # neurone 2 : mots négatifs
        if detail:
            print(f"{mot:<10} x = {x}   →   h1 (positif) = {h1:.3f}   h2 (négatif) = {h2:.3f}")
    return h1, h2

h = lire_avis("gâteau délicieux mais service lent", detail=True)
print("Mémoire finale :", h)

gâteau     x = [0.0, 0.0]   →   h1 (positif) = 0.000   h2 (négatif) = 0.000
délicieux  x = [1.0, 0.0]   →   h1 (positif) = 1.000   h2 (négatif) = 0.000
mais       x = [0.0, 0.0]   →   h1 (positif) = 0.500   h2 (négatif) = 0.000
service    x = [0.0, 0.0]   →   h1 (positif) = 0.250   h2 (négatif) = 0.000
lent       x = [0.0, 1.0]   →   h1 (positif) = 0.125   h2 (négatif) = 1.000
Mémoire finale : (0.125, 1.0)


**Lecture.** « délicieux » allume le neurone 1 ($h_1 = 1$), puis sa trace s’efface de moitié à chaque mot (0,5 → 0,25 → 0,125). « lent », lu en dernier, met $h_2 = 1$. À la fin, le neurone « négatif » domine : $h = (0{,}125 ; 1)$.

### Question 8 — Entraîner la sortie sur les 6 avis (Python pur)
**Réponse.** Chaque avis devient sa mémoire finale $(h_1, h_2)$. La sortie $p = \sigma(v_1 h_1 + v_2 h_2 + c)$ s’entraîne comme une régression logistique à 2 entrées (TP02), avec les 6 avis dans un seul batch, $\eta = 1$ et 100 epochs.

Epoch 1 à la main : au départ $p = 0{,}5$ pour tous les avis, donc $\delta = p - y = \pm 0{,}5$. On trouve $\nabla v_1 = -0{,}25$, $\nabla v_2 = +0{,}25$, $\nabla c = 0$, d’où $v_1 = 0{,}25$ et $v_2 = -0{,}25$ : dès la première epoch, le neurone 1 vote « positif » et le neurone 2 « négatif ».

In [ ]:
avis = ["gâteau délicieux", "service lent", "prix cher mais gâteau délicieux",
        "gâteau délicieux mais service lent", "accueil sympa et pain frais", "pain sec et prix cher"]
y_avis = [1, 0, 1, 0, 1, 0]
memoires = [lire_avis(a) for a in avis]

def entrainer_avis(H, Y, eta=1.0, epochs=100):
    v1, v2, c = 0.0, 0.0, 0.0
    historique = [(v1, v2, c)]
    for _ in range(epochs):
        g1 = g2 = gc = 0.0
        for (h1, h2), y in zip(H, Y):
            p = sigmoide(v1 * h1 + v2 * h2 + c)
            delta = p - y
            g1 += delta * h1;  g2 += delta * h2;  gc += delta
        n = len(H)
        v1, v2, c = v1 - eta * g1 / n, v2 - eta * g2 / n, c - eta * gc / n
        historique.append((v1, v2, c))
    return historique

hist_avis = entrainer_avis(memoires, y_avis)
v1, v2, c_avis = hist_avis[-1]
print(f"Après l'epoch 1   : v1 = {hist_avis[1][0]:.3f} ; v2 = {hist_avis[1][1]:.3f} ; c = {hist_avis[1][2]:.3f}")
print(f"Après 100 epochs  : v1 = {v1:.3f} ; v2 = {v2:.3f} ; c = {abs(c_avis):.3f}")
print()
p_avis = []
for a, (h1, h2), y in zip(avis, memoires, y_avis):
    p = sigmoide(v1 * h1 + v2 * h2 + c_avis)
    p_avis.append(p)
    print(f"{a:<36} h = ({h1:.3f} ; {h2:.3f})   p = {p:.2f}   prédit {int(p >= 0.5)}   réel {y}")

Après l'epoch 1   : v1 = 0.250 ; v2 = -0.250 ; c = 0.000
Après 100 epochs  : v1 = 3.883 ; v2 = -3.883 ; c = 0.000

gâteau délicieux                     h = (1.000 ; 0.000)   p = 0.98   prédit 1   réel 1
service lent                         h = (0.000 ; 1.000)   p = 0.02   prédit 0   réel 0
prix cher mais gâteau délicieux      h = (1.000 ; 0.125)   p = 0.97   prédit 1   réel 1
gâteau délicieux mais service lent   h = (0.125 ; 1.000)   p = 0.03   prédit 0   réel 0
accueil sympa et pain frais          h = (1.125 ; 0.000)   p = 0.99   prédit 1   réel 1
pain sec et prix cher                h = (0.000 ; 1.125)   p = 0.01   prédit 0   réel 0


### Question 9 — L’ordre des mots compte
**Réponse.** On donne au modèle les **mêmes mots dans un ordre différent**.

In [ ]:
nouveaux = ["délicieux mais lent", "lent mais délicieux",
            "service sympa mais gâteau sec", "gâteau sec mais service sympa"]
for a in nouveaux:
    h1, h2 = lire_avis(a)
    p = sigmoide(v1 * h1 + v2 * h2 + c_avis)
    print(f"{a:<32} h = ({h1:.3f} ; {h2:.3f})   p = {p:.2f} → {'positif' if p >= 0.5 else 'négatif'}")

délicieux mais lent              h = (0.250 ; 1.000)   p = 0.05 → négatif
lent mais délicieux              h = (1.000 ; 0.250)   p = 0.95 → positif
service sympa mais gâteau sec    h = (0.125 ; 1.000)   p = 0.03 → négatif
gâteau sec mais service sympa    h = (1.000 ; 0.125)   p = 0.97 → positif


**Lecture.** « délicieux mais lent » est négatif, « lent mais délicieux » est positif : mêmes mots, autre ordre, autre réponse. Un modèle qui compterait seulement les mots (sans ordre) donnerait la même réponse aux deux. C’est la **mémoire** du RNN, qui donne plus de poids aux derniers mots, qui fait la différence.

### Question 10 — Le même modèle de texte dans Keras
**Réponse.** On reproduit toute la chaîne de traitement du texte :
1. **tokenisation** : chaque mot reçoit un numéro (0 est réservé au remplissage) ;
2. **padding** : les avis n’ont pas la même longueur, on complète **au début** avec des 0 pour avoir des séquences de même longueur ;
3. `Embedding` (gelé) : numéro → vecteur $[\text{positif} ; \text{négatif}]$, le mot de remplissage donne $[0 ; 0]$ ;
4. `SimpleRNN(2)` (gelé) : les deux neurones, avec $W_x$ et $W_h$ ci-dessus. Comme les 0 du début laissent la mémoire à 0, le remplissage ne change rien ;
5. `Dense(1, sigmoid)` : la sortie, entraînée avec `SGD(learning_rate=1)`, 100 epochs, les 6 avis dans un batch.

In [ ]:
# 1) tokenisation : un numéro par mot (0 = remplissage)
vocabulaire = sorted({mot for a in avis + nouveaux for mot in a.split()})
numero = {mot: i + 1 for i, mot in enumerate(vocabulaire)}

# 2) padding au début
def encoder(liste_avis, longueur=5):
    lignes = []
    for a in liste_avis:
        ids = [numero[m] for m in a.split()]
        lignes.append([0] * (longueur - len(ids)) + ids)
    return np.array(lignes, dtype="int32")

X_avis = encoder(avis)
print("Avis 4 encodé :", X_avis[3], "←", avis[3])

# 3) matrice d'embedding : ligne i = vecteur du mot numéro i
matrice = np.zeros((len(vocabulaire) + 1, 2))
for mot, i in numero.items():
    matrice[i] = embedding(mot)

def creer_modele_texte():
    emb = layers.Embedding(len(vocabulaire) + 1, 2, trainable=False)
    rnn = layers.SimpleRNN(2, activation=None, trainable=False)
    modele = keras.Sequential([keras.Input(shape=(5,), dtype="int32"), emb, rnn,
                               layers.Dense(1, activation="sigmoid",
                                            kernel_initializer="zeros", bias_initializer="zeros")])
    emb.set_weights([matrice])
    rnn.set_weights([np.eye(2),              # W_x : neurone 1 ← positif, neurone 2 ← négatif
                     0.5 * np.eye(2),        # W_h : chaque neurone garde la moitié de SA mémoire
                     np.zeros(2)])           # b
    modele.compile(optimizer=keras.optimizers.SGD(learning_rate=1.0), loss="binary_crossentropy")
    return modele, rnn

modele_texte, rnn_texte = creer_modele_texte()
lecteur = keras.Sequential([modele_texte.layers[0], modele_texte.layers[1]])   # embedding + RNN
H_keras = keras.ops.convert_to_numpy(lecteur(X_avis))
print("Mémoires (h1, h2) calculées par Keras :")
print(H_keras.round(3))

modele_texte.fit(X_avis, np.array(y_avis, dtype="float32").reshape(-1, 1),
                 epochs=100, batch_size=6, shuffle=False, verbose=0)
k, b = modele_texte.layers[-1].get_weights()
v_keras = (float(k[0, 0]), float(k[1, 0]), float(b[0]))
p_keras_avis = modele_texte.predict(X_avis, verbose=0).ravel()
p_keras_nouveaux = modele_texte.predict(encoder(nouveaux), verbose=0).ravel()
print(f"Sortie Keras après 100 epochs : v1 = {v_keras[0]:.3f} ; v2 = {v_keras[1]:.3f} ; c = {abs(v_keras[2]):.3f}")
print("p sur les 4 nouveaux avis :", p_keras_nouveaux.round(2))

Avis 4 encodé : [ 6  3  8 12  7] ← gâteau délicieux mais service lent


Mémoires (h1, h2) calculées par Keras :
[[1.    0.   ]
 [0.    1.   ]
 [1.    0.125]
 [0.125 1.   ]
 [1.125 0.   ]
 [0.    1.125]]


Sortie Keras après 100 epochs : v1 = 3.883 ; v2 = -3.883 ; c = 0.000
p sur les 4 nouveaux avis : [0.05 0.95 0.03 0.97]


### Question 11 — Comparaison et validation de l’exemple 3

In [ ]:
comp = pd.DataFrame({
    "Avis": avis,
    "h1 Python pur": [m[0] for m in memoires], "h1 Keras": H_keras[:, 0],
    "h2 Python pur": [m[1] for m in memoires], "h2 Keras": H_keras[:, 1],
    "p Python pur": p_avis, "p Keras": p_keras_avis})
display(comp.round(3))

assert np.allclose(memoires, H_keras, atol=1e-5)
assert np.allclose((v1, v2, c_avis), v_keras, atol=1e-3)
assert np.allclose(p_avis, p_keras_avis, atol=1e-3)
p_pur_nouveaux = [sigmoide(v1 * h1 + v2 * h2 + c_avis) for h1, h2 in map(lire_avis, nouveaux)]
assert np.allclose(p_pur_nouveaux, p_keras_nouveaux, atol=1e-3)
print("✅ Exemple 3 : Python pur = Keras (mémoires des 2 neurones, poids de sortie, probabilités)")

,Avis,h1 Python pur,h1 Keras,h2 Python pur,h2 Keras,p Python pur,p Keras
0,gâteau délicieux,1.000,1.000,0.000,0.000,0.980,0.980
1,service lent,0.000,0.000,1.000,1.000,0.020,0.020
2,prix cher mais gâteau délicieux,1.000,1.000,0.125,0.125,0.968,0.968
3,gâteau délicieux mais service lent,0.125,0.125,1.000,1.000,0.032,0.032
4,accueil sympa et pain frais,1.125,1.125,0.000,0.000,0.987,0.987
5,pain sec et prix cher,0.000,0.000,1.125,1.125,0.013,0.013


✅ Exemple 3 : Python pur = Keras (mémoires des 2 neurones, poids de sortie, probabilités)


## Conclusion

| Ce qu’on a comparé (Python pur ↔ Keras) | Résultat |
|---|---|
| Mémoire $h$ après lecture de lundi, mardi, mercredi | ✅ identique (4,25) |
| Exemple 1 (ventes de jeudi, régression) : $v$ et $c$ à chaque epoch, prévision 395 TND | ✅ identiques |
| Exemple 2 (ventes en hausse ?, classification) : $v$ et $c$ à chaque epoch | ✅ identiques |
| Exemple 3 (avis clients, 2 neurones) : mémoires $h_1, h_2$, poids de sortie, probabilités | ✅ identiques |

**À retenir**
- Un RNN lit la séquence pas à pas avec les **mêmes poids** à chaque pas ; $h_t$ est sa mémoire.
- Un RNN traite toute séquence : texte (chaque mot devient un vecteur, l’embedding), séries temporelles, signaux.
- Avec plusieurs neurones, chaque neurone garde **sa** partie de la mémoire (ici : « positif » et « négatif ») et la sortie les combine.
- L’ordre compte : « délicieux mais lent » et « lent mais délicieux » ne donnent pas la même réponse.
- La même mémoire peut alimenter une sortie de **régression** (linéaire + MSE) ou de **classification** (sigmoïde + log-loss).
- À chaque pas, l’ancienne mémoire est multipliée par $W_h$ : sur de longues séquences, le début s’efface et le gradient **s’évanouit**. C’est ce que corrigent le LSTM et le GRU.